# BasketTube - AI Game Analyzer


This notebook implements:
- **Task 1**: Analyzing player performance from video commentary using whisper
- **Task 2**: Verifying player actions from video footage
  - Task 2.1: Chunking video into plays
  - Task 2.2: Analyzing specific player actions with timestamps (bird eyes view not working so I did not include)

## Setup and Installation

In [47]:
# Install required packages
!pip install -q openai-whisper
!pip install -q anthropic
!pip install -q moviepy
!pip install -q numpy
!pip install -q pillow
!pip install -q ultralytics

In [48]:
import whisper
import cv2
import numpy as np
import anthropic
import base64
import json
from pathlib import Path
from moviepy.editor import VideoFileClip
from io import BytesIO
from PIL import Image
from typing import List
from dataclasses import dataclass

## Configuration

In [49]:
from google.colab import userdata

# Anthropic API Key
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')

# Access video
VIDEO_PATH = Path("/content/drive/MyDrive/Basketball.mp4")

# Create output directory
OUTPUT_DIR = Path("basketball_analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Task 1: Analyzing Player Performance from Commentary

## Step 1.1: Extract Audio and Transcribe Commentary

In [50]:
class BasketballAnalyzer:
    """Complete basketball game analyzer."""

    def __init__(self, video_path, output_dir, api_key):
        self.video_path = str(video_path)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.api_key = api_key

        print("Initializing Basketball Analyzer...")
        print("Loading Whisper model...")
        self.whisper_model = whisper.load_model("base")

        print("Initializing Claude client...")
        self.claude_client = anthropic.Anthropic(api_key=api_key)

        self.transcription = None

    def extract_and_transcribe(self):
        """Extract audio and transcribe with timestamps."""
        print("\n" + "="*60)
        print("EXTRACTING AND TRANSCRIBING AUDIO")
        print("="*60)

        # Check if transcription already exists
        transcription_path = self.output_dir / "transcription.json"
        if transcription_path.exists():
            print("Loading existing transcription...")
            with open(transcription_path, 'r') as f:
                self.transcription = json.load(f)
            print(f"[OK] Loaded {len(self.transcription['segments'])} segments")
            return self.transcription

        print("Extracting audio from video...")
        video = VideoFileClip(self.video_path)

        audio_path = self.output_dir / "temp_audio.wav"
        video.audio.write_audiofile(str(audio_path), verbose=False, logger=None)
        video.close()
        print(f"[OK] Audio extracted to {audio_path}")

        print("Transcribing commentary (this may take a few minutes)...")
        self.transcription = self.whisper_model.transcribe(
            str(audio_path),
            word_timestamps=True,
            verbose=False
        )

        print(f"[OK] Transcribed {len(self.transcription['segments'])} segments")

        # Save transcription
        with open(transcription_path, 'w') as f:
            json.dump(self.transcription, f, indent=2)
        print(f"[OK] Transcription saved to {transcription_path}")

        return self.transcription

    def get_formatted_transcript(self):
        """Get formatted transcript with timestamps."""
        if not self.transcription:
            raise ValueError("No transcription available. Run extract_and_transcribe() first.")

        formatted = []
        for segment in self.transcription['segments']:
            timestamp = self._format_time(segment['start'])
            formatted.append(f"({timestamp}): {segment['text']}")
        return "\n".join(formatted)

    def analyze_commentary(self, query):
        """Analyze commentary using Claude."""
        if not self.transcription:
            raise ValueError("No transcription available. Run extract_and_transcribe() first.")

        print(f"\nAnalyzing query: '{query}'")

        formatted_transcript = self.get_formatted_transcript()

        system_prompt = """You are an expert basketball analyst with access to game commentary.
        Answer queries about player performance, citing specific timestamps.
        Format timestamps as (MM:SS)."""

        message = self.claude_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=2000,
            system=system_prompt,
            messages=[{
                "role": "user",
                "content": f"Commentary:\n{formatted_transcript}\n\nQuery: {query}"
            }]
        )

        return message.content[0].text

    @staticmethod
    def _format_time(seconds):
        """Convert seconds to MM:SS format."""
        minutes = int(seconds // 60)
        secs = int(seconds % 60)
        return f"{minutes:02d}:{secs:02d}"

In [51]:
print("\n" + "="*70)
print("TASK 1: ANALYZING PLAYER PERFORMANCE FROM COMMENTARY")
print("="*70)

# Initialize analyzer
analyzer = BasketballAnalyzer(
    video_path=VIDEO_PATH,
    output_dir=OUTPUT_DIR,
    api_key=ANTHROPIC_API_KEY
)

# Extract and transcribe
transcription_result = analyzer.extract_and_transcribe()

# Show first few segments
print("\nFirst few commentary segments:")
for i, segment in enumerate(transcription_result['segments'][:5], 1):
    print(f"{i}. [{segment['start']:.2f}s - {segment['end']:.2f}s]: {segment['text']}")

# Example query
query = "Analyze the player that scored the most in this game"
print(f"\nQuery: {query}")
print("\nAnalysis:")
analysis = analyzer.analyze_commentary(query)
print(analysis)


TASK 1: ANALYZING PLAYER PERFORMANCE FROM COMMENTARY
Initializing Basketball Analyzer...
Loading Whisper model...
Initializing Claude client...

EXTRACTING AND TRANSCRIBING AUDIO
Loading existing transcription...
[OK] Loaded 1356 segments

First few commentary segments:
1. [0.00s - 4.54s]:  I was not breaking news. They're a different basketball team with a healthy whole LeBron James
2. [5.64s - 9.82s]:  15 games over 500 when he's healthy three games under 500
3. [10.48s - 14.06s]:  Creates offense for himself and his teammates. They need him
4. [14.06s - 19.80s]:  They all Jeff Steph Curry what he's done over the past month and a half has been absolutely brilliant
5. [19.80s - 26.84s]:  He says valuable as any player in this league they play great defense and they have the ultimate home run hitter

Query: Analyze the player that scored the most in this game

Analysis:

Analyzing query: 'Analyze the player that scored the most in this game'
Based on the commentary, **Steph Curry scor

In [52]:
def chat_interface(analyzer):
    """Simple chat interface for querying game commentary."""
    # Check if transcription exists
    if not analyzer.transcription:
        print("Error: No transcription available.")
        print("Please run: analyzer.extract_and_transcribe() first")
        return

    print("=" * 60)
    print("Basketball Commentary Analysis Chat")
    print("=" * 60)
    print("Ask questions about player performance based on commentary.")
    print("Type 'quit' or 'exit' to end the session.\n")

    while True:
        query = input("\nYour question: ").strip()

        if query.lower() in ['quit', 'exit', 'q']:
            print("Ending analysis session. Goodbye!")
            break

        if not query:
            print("Please enter a question.")
            continue

        print("\nAnalyzing...\n")
        response = analyzer.analyze_commentary(query)
        print(response)
        print("\n" + "-" * 60)


chat_interface(analyzer)

Basketball Commentary Analysis Chat
Ask questions about player performance based on commentary.
Type 'quit' or 'exit' to end the session.


Your question: Who is the best players on either team?

Analyzing...


Analyzing query: 'Who is the best players on either team?'
Based on the game commentary, the best players on either team are clearly:

**Los Angeles Lakers:**
- **LeBron James** - Described as creating offense for himself and teammates, and the commentary notes "They need him" (00:10). He finished with 22 points, 11 rebounds, 10 assists and hit the game-winning three-pointer (91:12-91:16)
- **Anthony Davis** - Had a difficult first half but came alive in the second half with 25 points and 12 rebounds, including 13 points in the fourth quarter (86:03-86:05)

**Golden State Warriors:**
- **Stephen Curry** - The commentary emphasizes his brilliance: "what he's done over the past month and a half has been absolutely brilliant" and calls him "as valuable as any player in this league"

# Task 2: Verifying Player Actions from Video

In [53]:
@dataclass
class CommentaryEvent:
    """Represents an event mentioned in commentary"""
    timestamp: str  # Format: "MM:SS"
    timestamp_seconds: float
    player: str
    action: str
    description: str

class WorkingEventExtractor:
    """Extract events from transcription_result with segments"""

    def __init__(self, transcription_result: Dict):
        """
        Args:
            transcription_result: Dict with 'segments' key
        """
        self.transcription = transcription_result
        self.segments = transcription_result.get('segments', [])
        print(f"Loaded {len(self.segments)} segments")

    def extract_events(self) -> List[CommentaryEvent]:
        """Extract events using actual timestamps from segments"""
        events = []

        action_keywords = {
            'shoot': 'Shooting',
            'shot': 'Shooting',
            'shoots': 'Shooting',
            'three': '3-Point Shot',
            'layup': 'Layup',
            'dunk': 'Dunk',
            'dunks': 'Dunk',
            'score': 'Scoring',
            'scores': 'Scoring',
            'scored': 'Scoring',
            'bucket': 'Scoring',
            'assist': 'Assist',
            'assists': 'Assist',
            'pass': 'Passing',
            'passes': 'Passing',
            'rebound': 'Rebounding',
            'rebounds': 'Rebounding',
            'block': 'Block',
            'blocks': 'Block',
            'blocked': 'Block',
            'steal': 'Steal',
            'steals': 'Steal',
            'drive': 'Driving',
            'drives': 'Driving',
            'make': 'Scoring',
            'makes': 'Scoring',
            'made': 'Scoring'
        }

        players = [
            'Curry', 'Stephen Curry', 'Steph',
            'LeBron', 'LeBron James', 'James',
            'Davis', 'Anthony Davis',
            'Wiggins', 'Andrew Wiggins',
            'Green', 'Draymond Green', 'Draymond', 'Dremond',
            'Schröder', 'Dennis Schröder', 'Schroder', 'Shruder',
            'Pope', 'Kentavious Caldwell-Pope',
            'Drummond', 'Andre Drummond',
            'Poole', 'Jordan Poole',
            'Caruso', 'Alex Caruso',
            'Looney', 'Kevon Looney',
            'Bazemore', 'Kent Bazemore',
            'Harrell', 'Montrezl Harrell',
            'Kuzma', 'Kyle Kuzma'
        ]

        print(f"Scanning {len(self.segments)} segments for events...")

        for segment in self.segments:
            start_time = float(segment.get('start', 0))
            text = segment.get('text', '').strip()
            text_lower = text.lower()

            # Look for player + action combinations
            for player in players:
                if player.lower() in text_lower:
                    for keyword, action in action_keywords.items():
                        if keyword in text_lower:
                            minutes = int(start_time // 60)
                            seconds = int(start_time % 60)

                            events.append(CommentaryEvent(
                                timestamp=f"{minutes}:{seconds:02d}",
                                timestamp_seconds=start_time,
                                player=player,
                                action=action,
                                description=text
                            ))
                            break  # One action per segment per player

        # Remove duplicates (same timestamp within 2 seconds + same player + action)
        unique_events = []
        seen = set()
        for event in events:
            # Round to nearest 2 seconds for deduplication
            rounded_time = round(event.timestamp_seconds / 2) * 2
            key = (rounded_time, event.player, event.action)
            if key not in seen:
                seen.add(key)
                unique_events.append(event)

        unique_events.sort(key=lambda x: x.timestamp_seconds)

        print(f"Found {len(unique_events)} unique events")
        return unique_events

In [54]:
print("="*60)
print("EXTRACTING EVENTS")
print("="*60)

extractor = WorkingEventExtractor(transcription_result)
events = extractor.extract_events()

print(f"\nFound {len(events)} events")
print("\nFirst 15 events:")
for i, event in enumerate(events[:15]):
    print(f"{i+1:2d}. {event.timestamp:>6s} - {event.player:20s} - {event.action}")
    print(f"    '{event.description[:80]}...'")

if len(events) > 15:
    print(f"\n... and {len(events) - 15} more events")

# Some statistics
print("\n" + "="*60)
print("EVENT STATISTICS")
print("="*60)

# Events by player
from collections import Counter
player_counts = Counter([e.player for e in events])
print("\nEvents by player:")
for player, count in player_counts.most_common(10):
    print(f"  {player:20s}: {count:3d} events")

# Events by action
action_counts = Counter([e.action for e in events])
print("\nEvents by action:")
for action, count in action_counts.most_common():
    print(f"  {action:20s}: {count:3d} events")

# Time distribution
print(f"\nTime range:")
print(f"  First event: {events[0].timestamp}")
print(f"  Last event:  {events[-1].timestamp}")
print(f"  Duration:    {events[-1].timestamp_seconds / 60:.1f} minutes")

EXTRACTING EVENTS
Loaded 1356 segments
Scanning 1356 segments for events...
Found 257 unique events

Found 257 events

First 15 events:
 1.   0:50 - Davis                - Shooting
    'And here's one of the fun matchups in this series Anthony Davis has to take a to...'
 2.   0:50 - Anthony Davis        - Shooting
    'And here's one of the fun matchups in this series Anthony Davis has to take a to...'
 3.   0:50 - Green                - Shooting
    'And here's one of the fun matchups in this series Anthony Davis has to take a to...'
 4.   0:50 - Dremond              - Shooting
    'And here's one of the fun matchups in this series Anthony Davis has to take a to...'
 5.   1:06 - Wiggins              - Scoring
    'And as Andrew Wiggins gets inside and gets the bucket...'
 6.   1:06 - Andrew Wiggins       - Scoring
    'And as Andrew Wiggins gets inside and gets the bucket...'
 7.   2:16 - Curry                - Scoring
    'He didn't wait for step-curry to make a play waived everybody

In [55]:
class VerificationAnalyzer:
    """
    Analyzes video to verify what commentary claims
    """

    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = YOLO('yolo11n.pt')

    def frame_to_base64(self, frame: np.ndarray) -> str:
        """Convert frame to base64"""
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        pil_image.thumbnail((800, 600))
        buffered = BytesIO()
        pil_image.save(buffered, format="JPEG", quality=85)
        return base64.b64encode(buffered.getvalue()).decode('utf-8')

    def verify_event(self, play: TargetedPlay) -> Dict:
        """
        Verify if the claimed event actually happened in the video

        Returns:
            {
                'verified': bool,
                'confidence': float,
                'observations': list of observations,
                'verdict': string
            }
        """
        event = play.event

        print(f"\nVerifying: {event.player} - {event.action} at {event.timestamp}")

        observations = []

        # Analyze middle frame (closest to event timestamp)
        middle_idx = len(play.frames) // 2
        if middle_idx >= len(play.frames):
            middle_idx = len(play.frames) - 1

        key_frame = play.frames[middle_idx]

        # Get YOLO detections
        results = self.model(key_frame, classes=[0], verbose=False)
        player_count = len(results[0].boxes) if len(results) > 0 and results[0].boxes is not None else 0

        observations.append(f"{player_count} players detected in frame")

        # Analyze with Claude
        frame_base64 = self.frame_to_base64(key_frame)

        prompt = f"""You are verifying a basketball commentary claim against actual video footage.

                CLAIM FROM COMMENTARY:
                - Player: {event.player}
                - Action: {event.action}
                - Time: {event.timestamp}
                - Description: {event.description}

                YOUR TASK:
                Analyze this video frame and determine if the claimed action is visible or plausible.

                Look for:
                1. Is {event.player} visible in the frame?
                2. Is the action "{event.action}" happening or about to happen?
                3. Are there signs this is the described play?

                Respond with ONLY valid JSON (no markdown):
                {{
                    "player_visible": true/false,
                    "action_matches": true/false,
                    "confidence": 0.0-1.0,
                    "what_you_see": "detailed description",
                    "verified": true/false
                }}"""

        try:
            response = self.client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=500,
                messages=[{
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/jpeg",
                                "data": frame_base64
                            }
                        },
                        {
                            "type": "text",
                            "text": prompt
                        }
                    ]
                }]
            )

            response_text = response.content[0].text.strip()
            if "```" in response_text:
                response_text = response_text.split("```")[1]
                if response_text.startswith("json"):
                    response_text = response_text[4:].strip()

            result = json.loads(response_text)

            observations.append(result.get('what_you_see', 'No description'))

            verified = result.get('verified', False)
            confidence = result.get('confidence', 0.0)

            if verified:
                verdict = f"VERIFIED: {event.action} by {event.player}"
            else:
                verdict = f"NOT VERIFIED: Could not confirm {event.action}"

            return {
                'verified': verified,
                'confidence': confidence,
                'observations': observations,
                'verdict': verdict,
                'frame': key_frame
            }

        except Exception as e:
            print(f"Error: {e}")
            return {
                'verified': False,
                'confidence': 0.0,
                'observations': [f"Analysis error: {e}"],
                'verdict': "UNABLE TO VERIFY",
                'frame': key_frame
            }

In [56]:
# Save events
events_data = [
    {
        'timestamp': e.timestamp,
        'timestamp_seconds': e.timestamp_seconds,
        'player': e.player,
        'action': e.action,
        'description': e.description
    }
    for e in events
]

import json
events_path = OUTPUT_DIR / "extracted_events.json"
with open(events_path, 'w') as f:
    json.dump(events_data, f, indent=2)

print(f"\nEvents saved to: {events_path}")



Events saved to: basketball_analysis_output/extracted_events.json


In [57]:
print("\nStep 3: Extracting video segments around events...")
print(f"Context window: ±10 seconds around each event")

detector = CommentaryGuidedPlayDetector(VIDEO_PATH)
plays = detector.extract_plays_from_events(events, context_window=10.0)

print(f"\nExtracted {len(plays)} video segments")


Step 3: Extracting video segments around events...
Context window: ±10 seconds around each event
Extracting 257 plays from commentary events...
  Play 1: Davis - Shooting at 0:50
  Play 2: Anthony Davis - Shooting at 0:50
  Play 3: Green - Shooting at 0:50
  Play 4: Dremond - Shooting at 0:50
  Play 5: Wiggins - Scoring at 1:06
  Play 6: Andrew Wiggins - Scoring at 1:06
  Play 7: Curry - Scoring at 2:16
  Play 8: Curry - Passing at 3:20
  Play 9: Davis - Passing at 3:20
  Play 10: Anthony Davis - Passing at 3:20
  Play 11: Drummond - Rebounding at 3:29
  Play 12: Curry - Shooting at 4:00
  Play 13: Green - Shooting at 4:00
  Play 14: Dremond - Shooting at 4:00
  Play 15: Looney - Shooting at 4:00
  Play 16: James - Block at 4:13
  Play 17: Davis - Block at 4:13
  Play 18: Curry - Shooting at 4:24
  Play 19: Looney - Shooting at 4:24
  Play 20: James - Block at 5:25
  Play 21: Wiggins - Block at 5:25
  Play 22: Andrew Wiggins - Block at 5:25
  Play 23: James - 3-Point Shot at 5:58
  Pl

In [61]:
print("\nStep 4: Verifying commentary claims against video...")

analyzer = VerificationAnalyzer(ANTHROPIC_API_KEY)

verification_results = []

# Limit to first 25 plays
plays_to_analyze = plays[:25]

for i, play in enumerate(plays_to_analyze, 1):
    print(f"Analyzing play {i}/25...")
    result = analyzer.verify_event(play)
    verification_results.append({
        'timestamp': play.event.timestamp,
        'player': play.event.player,
        'claimed_action': play.event.action,
        'verified': result['verified'],
        'confidence': result['confidence'],
        'observations': '; '.join(result['observations']),
        'verdict': result['verdict']
    })

    print(f"  {result['verdict']} (confidence: {result['confidence']:.2f})")

print(f"\nCompleted verification of {len(verification_results)} plays")


Step 4: Verifying commentary claims against video...
Analyzing play 1/25...

Verifying: Davis - Shooting at 0:50
  NOT VERIFIED: Could not confirm Shooting (confidence: 0.70)
Analyzing play 2/25...

Verifying: Anthony Davis - Shooting at 0:50
  VERIFIED: Shooting by Anthony Davis (confidence: 0.85)
Analyzing play 3/25...

Verifying: Green - Shooting at 0:50
  NOT VERIFIED: Could not confirm Shooting (confidence: 0.70)
Analyzing play 4/25...

Verifying: Dremond - Shooting at 0:50
  NOT VERIFIED: Could not confirm Shooting (confidence: 0.70)
Analyzing play 5/25...

Verifying: Wiggins - Scoring at 1:06
  NOT VERIFIED: Could not confirm Scoring (confidence: 0.80)
Analyzing play 6/25...

Verifying: Andrew Wiggins - Scoring at 1:06
  NOT VERIFIED: Could not confirm Scoring (confidence: 0.20)
Analyzing play 7/25...

Verifying: Curry - Scoring at 2:16
  VERIFIED: Scoring by Curry (confidence: 0.75)
Analyzing play 8/25...

Verifying: Curry - Passing at 3:20
  NOT VERIFIED: Could not confirm Pa